System construction and test


In [8]:
from datetime import date, datetime
import pandas as pd
import yfinance as yf
import time
import numpy as np
pd.options.mode.chained_assignment = None  # default='warn'


input:
    
     Titulo  : example: GGAL
     frequencia de tick : 1h (frequencia mais alta)
     dftitulo : vista do BD sqtitulosalpha.bd (testar ultimos periodas)
Parametros a variar para back testing ( definir rangos de variação)
     k, d, smooth (parametros do stch)
     dayM  (frequencia media) (multiplicador da frequencia mais alta) exemplo: 5
     semM  (frequencia baixa) (multiplicador da frequencia media) exemplo: 7

In [13]:
dataini = '2014-04-03 19:30:00'
datafim = '2024-04-03 19:30:00'

In [15]:
#%%timeit
import sqlite3
import pandas as pd

# Caminho para o banco de dados
caminho_bd = r'C:\Users\scitr\anaconda_projects\Trading_System\Dados_Fontes\Alpha_Vantage\sqtitulosalpha.db'

# Conectando ao banco
conexao = sqlite3.connect(caminho_bd)

# Lendo a view
#consulta = 'SELECT * FROM vwtitulosdados ORDER BY datetime'
consulta = f"""
SELECT * FROM vwtitulosdados
WHERE datetime BETWEEN '{dataini}' AND '{datafim}'
ORDER BY datetime
"""

dftitulosdados = pd.read_sql_query(consulta, conexao)

# Fechando a conexão
conexao.close()

# Exibindo os primeiros registros para conferir
#display(dftitulosdados)
dftitulosdados = dftitulosdados.drop(columns=["symbol", "moeda", "intervalo","volume"])
display(len(dftitulosdados))
#display(dftitulosdados.head(10))

21365

In [17]:

i = 'high'
K = 8
D = 6   
smoth = 3 
medM = 5 
lowM = 30
stpl = 0.025
comission = 0.0035
taxalivrerisgoprom = 0.05
drawmax = -0.1


In [19]:
#%%timeit
# Stochastic calculation
def stochastic(dftitulosdados, i, K, D, smoth):
    df = dftitulosdados    
    df["k"] = (100. * (df.close - df.low.rolling(K).min()) /
        (df.high.rolling(K).max() - df.low.rolling(K).min()))
    
    df["k" + i ] = df.k.rolling(smoth).mean()
    df["d" + i ] = df["k" + i].rolling(D).mean()
    
    df.drop(columns=["k"], inplace=True)  
    dfstoch = df
    return dfstoch



In [21]:
dfstoch = stochastic(dftitulosdados, i, K, D, smoth)
display (dfstoch.head(10))

,datetime,open,high,low,close,khigh,dhigh
0,2014-04-04 09:00:00,10.3862,10.7565,10.3862,10.7186,NaN,NaN
1,2014-04-04 10:00:00,10.6903,10.7292,10.4797,10.5265,NaN,NaN
2,2014-04-04 11:00:00,10.5343,10.6123,10.2848,10.3316,NaN,NaN
3,2014-04-04 12:00:00,10.3316,10.3472,10.0977,10.1444,NaN,NaN
4,2014-04-04 13:00:00,10.1444,10.2119,10.0977,10.1990,NaN,NaN
5,2014-04-04 14:00:00,10.2068,10.2458,10.1522,10.1756,NaN,NaN
6,2014-04-04 15:00:00,10.1756,10.2224,10.1210,10.1678,NaN,NaN
7,2014-04-04 16:00:00,10.1444,10.1444,10.1444,10.1444,NaN,NaN
8,2014-04-07 09:00:00,10.1600,10.1600,10.0197,10.0899,NaN,NaN
9,2014-04-07 10:00:00,10.1055,10.4251,10.0587,10.2848,20.572668,NaN


In [23]:
#%%timeit
# stochastic high, med and low frcuency

def stoch_hml( dfstoch, k, d, smth, medM, lowM): 
    df = dfstoch
    df = stochastic(df, "high", k, d, smth)
    df = stochastic(df, "med", k*medM, d*medM, smth*medM)
    df = stochastic(df, "low", k*medM*lowM, d*medM*lowM, smth*medM*lowM)
    dfstoch_hml = df
    return dfstoch_hml
    



In [25]:
dfstoch_hml= stoch_hml(dfstoch , K, D, smoth, medM , lowM )
display(dfstoch_hml.head(10) )

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow
0,2014-04-04 09:00:00,10.3862,10.7565,10.3862,10.7186,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-04-04 10:00:00,10.6903,10.7292,10.4797,10.5265,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-04-04 11:00:00,10.5343,10.6123,10.2848,10.3316,NaN,NaN,NaN,NaN,NaN,NaN
3,2014-04-04 12:00:00,10.3316,10.3472,10.0977,10.1444,NaN,NaN,NaN,NaN,NaN,NaN
4,2014-04-04 13:00:00,10.1444,10.2119,10.0977,10.1990,NaN,NaN,NaN,NaN,NaN,NaN
5,2014-04-04 14:00:00,10.2068,10.2458,10.1522,10.1756,NaN,NaN,NaN,NaN,NaN,NaN
6,2014-04-04 15:00:00,10.1756,10.2224,10.1210,10.1678,NaN,NaN,NaN,NaN,NaN,NaN
7,2014-04-04 16:00:00,10.1444,10.1444,10.1444,10.1444,NaN,NaN,NaN,NaN,NaN,NaN
8,2014-04-07 09:00:00,10.1600,10.1600,10.0197,10.0899,NaN,NaN,NaN,NaN,NaN,NaN
9,2014-04-07 10:00:00,10.1055,10.4251,10.0587,10.2848,20.572668,NaN,NaN,NaN,NaN,NaN


In [27]:
#%%timeit
def system_criterias (dfstoch_hml):   # input  df() =  dfstoch_hml ()
    df = dfstoch_hml
    df["longbuylow"] = ((df["klow"] > 20) & (df["klow"] > df["dlow"])).astype(int)
    df["longbuymed"] = ((df["kmed"] > 20) & (df["kmed"] > df["dmed"])).astype(int)
    df["longbuyhigh"] = ((df["khigh"] > 20) & (df["khigh"] > df["dhigh"])).astype(int)
    dfcriterias = df
    return dfcriterias


In [29]:
dfcriterias = system_criterias (dfstoch_hml)
display (dfcriterias.head(10))

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow,longbuylow,longbuymed,longbuyhigh
0,2014-04-04 09:00:00,10.3862,10.7565,10.3862,10.7186,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
1,2014-04-04 10:00:00,10.6903,10.7292,10.4797,10.5265,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
2,2014-04-04 11:00:00,10.5343,10.6123,10.2848,10.3316,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
3,2014-04-04 12:00:00,10.3316,10.3472,10.0977,10.1444,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
4,2014-04-04 13:00:00,10.1444,10.2119,10.0977,10.1990,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
5,2014-04-04 14:00:00,10.2068,10.2458,10.1522,10.1756,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
6,2014-04-04 15:00:00,10.1756,10.2224,10.1210,10.1678,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
7,2014-04-04 16:00:00,10.1444,10.1444,10.1444,10.1444,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
8,2014-04-07 09:00:00,10.1600,10.1600,10.0197,10.0899,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
9,2014-04-07 10:00:00,10.1055,10.4251,10.0587,10.2848,20.572668,NaN,NaN,NaN,NaN,NaN,0,0,0


dfsignals = system_signals(dfcriterias)
display (dfsignals.head(10))

In [32]:
##%%timeit
def system_signals (dfcriterias):
    df = dfcriterias
    n = len(df)
    state_array = np.full(n, "standby", dtype=object)  # inicializa com "standby"
    estado_anterior = "standby"

    high = df["longbuyhigh"].to_numpy()
    med = df["longbuymed"].to_numpy()
    low = df["longbuylow"].to_numpy()

    for i in range(1, n):
        if high[i] == 1 and med[i] == 1 and low[i] == 1 and estado_anterior == "standby":
            state_array[i] = "enter"
            estado_anterior = "enter"
        elif med[i] == 1 and low[i] == 1 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "stay"
            estado_anterior = "stay"
        elif high[i] == 1 and low[i] == 1 and med[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "stay"
            estado_anterior = "stay"
        elif low[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "out"
            estado_anterior = "out"
        elif high[i] == 0 and med[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "out"
            estado_anterior = "out"
        elif (low[i] == 0 or med[i] == 0) and estado_anterior == "out":
            state_array[i] = "standby"
            estado_anterior = "standby"
        else:
            state_array[i] = estado_anterior

    df["state"] = state_array
    dfsignals = df
    return df

In [34]:
dfsignals = system_signals (dfcriterias)
display (dfsignals.head(10))

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow,longbuylow,longbuymed,longbuyhigh,state
0,2014-04-04 09:00:00,10.3862,10.7565,10.3862,10.7186,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
1,2014-04-04 10:00:00,10.6903,10.7292,10.4797,10.5265,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
2,2014-04-04 11:00:00,10.5343,10.6123,10.2848,10.3316,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
3,2014-04-04 12:00:00,10.3316,10.3472,10.0977,10.1444,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
4,2014-04-04 13:00:00,10.1444,10.2119,10.0977,10.1990,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
5,2014-04-04 14:00:00,10.2068,10.2458,10.1522,10.1756,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
6,2014-04-04 15:00:00,10.1756,10.2224,10.1210,10.1678,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
7,2014-04-04 16:00:00,10.1444,10.1444,10.1444,10.1444,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
8,2014-04-07 09:00:00,10.1600,10.1600,10.0197,10.0899,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
9,2014-04-07 10:00:00,10.1055,10.4251,10.0587,10.2848,20.572668,NaN,NaN,NaN,NaN,NaN,0,0,0,standby


Stop Loss Reentry

In [37]:
#%%timeit
def stop_loss_reentry (dfsignals, stpl) :

    df = dfsignals[(dfsignals['state'] == 'enter') | (dfsignals['state'] == 'stay')]
    df = df.reset_index(drop=True)
    df = df.drop(columns=["open" ,	"high" , "low" , "khigh","dhigh","kmed" ,"dmed","klow","dlow"])
    #drop(columns=["symbol", "moeda", "intervalo","volume"])
    df["stpl"] = 0.0
    stoplossprice = 0.0
    lastlongbuyprice = 0.0

    for i in range(0, len(df)):
        
        if df.loc[i, "state"] == "enter" :
           stoplossprice = df.loc[i, "close"]
           df.loc[i,"stpl"] = df.loc[i, "close"] - stoplossprice * (1 - stpl)
           
        if df.loc[i, "state"]== "stay" :
           df.loc[i, "stpl"] = df.loc[i, "close"] - stoplossprice * (1- stpl)
            
           if df.loc[i, "stpl"] < 0.0 :
                df.loc[i, "state"] = "out"
               
           if df.loc[i, "stpl"] > 0.0 and   (df.loc[i-1, "state"] == "out" or df.loc[i-1, "state"] == "outstpl") :
                df.loc[i, "state"] = "enter"
               
           if df.loc[i, "stpl"] < 0.0 and   (df.loc[i-1, "state"] == "out" or df.loc[i-1, "state"] == "outstpl") :
                df.loc[i, "state"] = "outstpl"

    return df


In [39]:
dfstoploss = stop_loss_reentry (dfsignals, stpl)
display(dfstoploss.head(10))

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,stpl
0,2015-11-12 10:00:00,20.5336,1,1,1,enter,0.513340
1,2015-11-12 11:00:00,20.5883,1,1,1,stay,0.568040
2,2015-11-12 12:00:00,20.5336,1,1,1,stay,0.513340
3,2015-11-12 13:00:00,20.4400,1,1,1,stay,0.419740
4,2015-11-12 14:00:00,20.3541,1,1,1,stay,0.333840
5,2015-11-12 15:00:00,20.3775,1,1,0,stay,0.357240
6,2015-11-12 16:00:00,20.3619,1,1,0,stay,0.341640
7,2015-11-13 08:00:00,20.3775,1,1,0,stay,0.357240
8,2015-11-13 09:00:00,20.3931,1,1,0,stay,0.372840
9,2015-11-16 10:00:00,20.4321,1,1,1,enter,0.510803


In [41]:
#%%timeit
# preparar dataframe para calcular index 
def index_dataframe (dfsignals, dfstoploss) :
    # elimino colunas de dfsignals e filtro por os valores "out"
    dfsignalsdrop = dfsignals.drop(columns=["open" ,"high" , "low" , "khigh","dhigh","kmed" ,"dmed","klow","dlow"])
    dfsignalsout = dfsignalsdrop[(dfsignalsdrop['state'] == 'out')]

    # elimino a culuna stpl de dfstoploss e filtro os valores enter e out 
    dfstoplossdrop = dfstoploss.drop(columns=["stpl"]) 
    dfstoplossenterout = dfstoplossdrop[(dfstoplossdrop['state'] == 'enter') | (dfstoplossdrop['state'] == 'out')]

    # concatenar os dois df para ter o total dos signals enter e out
    dfsignalsenterout = pd.concat([dfsignalsout, dfstoplossenterout], ignore_index=True)

    # Ordenar pelo datetime e resetear o index
    dfsignalsenterout["datetime"] = pd.to_datetime(dfsignalsenterout["datetime"])
    dfsignalsenterout = dfsignalsenterout.sort_values("datetime").reset_index(drop=True)

    # limpar os out duplicados"out" por a saida anticipada do stoploss e reiniciar indice
    df = dfsignalsenterout
    cond = (df["state"] == "out")  & (df["state"].shift(1) == "out")
    dfsignalsentoutclean = df[~cond].reset_index(drop=True)
    return dfsignalsentoutclean


In [43]:
dfsignalsentoutclean = index_dataframe (dfsignals, dfstoploss)
display (dfsignalsentoutclean)

,datetime,close,longbuylow,longbuymed,longbuyhigh,state
0,2015-11-12 10:00:00,20.5336,1,1,1,enter
1,2015-11-13 10:00:00,20.2799,1,0,0,out
2,2015-11-16 10:00:00,20.4321,1,1,1,enter
3,2015-11-19 15:00:00,21.2754,1,0,0,out
4,2015-11-23 08:00:00,22.9149,1,1,1,enter
...,...,...,...,...,...,...
408,2024-03-13 09:00:00,21.7135,1,1,1,enter
409,2024-03-18 10:00:00,21.8522,1,0,0,out
410,2024-03-18 17:00:00,23.0815,1,1,1,enter
411,2024-03-25 11:00:00,24.1040,1,0,0,out


Index ,Trade, Index sin comission

In [46]:
#%%timeit
def index_trade(dfsignalsenteroutclean):
    df = dfsignalsentoutclean
    df ["index_sc"] = 100. 
    df ["trade"] = 0.
    df ["index"] = 100. *(1-comission) 
    for i in range(1, len(df)):      
                       
        if  df.loc[i, "state"] == "out" :
            df.loc[i, "index_sc"] = (((df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"])+1)* df.loc[i-1,"index_sc"]
            df.loc[i, "index"] = df.loc[i, "index_sc"]* (1-comission)
            df.loc[i, "trade"] = (df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"]
            
        if  df.loc[i, "state"] == "enter" :        
            df.loc[i, "index_sc"] =  df.loc[i-1, "index_sc"]
            df.loc[i, "index"] = df.loc[i, "index_sc"]* (1-comission)
    dfindex = df
    return dfindex


In [48]:
dfindex = index_trade(dfsignalsentoutclean)
display (dfindex.head(10))

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,index_sc,trade,index
0,2015-11-12 10:00:00,20.5336,1,1,1,enter,100.000000,0.000000,99.650000
1,2015-11-13 10:00:00,20.2799,1,0,0,out,98.764464,-0.012355,98.418788
2,2015-11-16 10:00:00,20.4321,1,1,1,enter,98.764464,0.000000,98.418788
3,2015-11-19 15:00:00,21.2754,1,0,0,out,102.840799,0.041273,102.480856
4,2015-11-23 08:00:00,22.9149,1,1,1,enter,102.840799,0.000000,102.480856
5,2015-11-23 09:00:00,20.9709,1,1,0,out,94.116234,-0.084836,93.786828
6,2015-11-30 15:00:00,19.6436,1,1,1,enter,94.116234,0.000000,93.786828
7,2015-12-02 09:00:00,19.1517,1,1,0,out,91.759448,-0.025041,91.438290
8,2015-12-02 13:00:00,19.1751,1,1,1,enter,91.759448,0.000000,91.438290
9,2015-12-03 09:00:00,19.1439,1,1,1,out,91.610145,-0.001627,91.289510


stop drawdawn

In [51]:
#%%timeit
def stop_drawdawn (dfindex, drawmax):
    df = dfindex
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df["index"] = pd.to_numeric(df["index"], errors="coerce")

    estado_corrigido = []
    pico_atual = df.loc[0, "index"]

    for i in range(len(df)):
        valor_index = df.loc[i, "index"]
        estado = df.loc[i, "state"]

        # Atualiza pico se houve recuperação
        if valor_index > pico_atual:
            pico_atual = valor_index

        # Calcula drawdown
        if pico_atual > 0:
            drawdown = (valor_index - pico_atual) / pico_atual
        else:
            drawdown = 0

        # Verifica se deve aplicar stopsys
        if estado == "out" and drawdown < drawmax:
            estado = "stopsys"
            pico_atual = valor_index  # reinicia ciclo a partir desse ponto

        estado_corrigido.append(estado)
    df["state"] = estado_corrigido
    dfindexdrawdawn = df
    return dfindexdrawdawn

In [53]:
dfindexdrawdawn = stop_drawdawn(dfindex, drawmax)
display(dfindexdrawdawn)

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,index_sc,trade,index
0,2015-11-12 10:00:00,20.5336,1,1,1,enter,100.000000,0.000000,99.650000
1,2015-11-13 10:00:00,20.2799,1,0,0,out,98.764464,-0.012355,98.418788
2,2015-11-16 10:00:00,20.4321,1,1,1,enter,98.764464,0.000000,98.418788
3,2015-11-19 15:00:00,21.2754,1,0,0,out,102.840799,0.041273,102.480856
4,2015-11-23 08:00:00,22.9149,1,1,1,enter,102.840799,0.000000,102.480856
...,...,...,...,...,...,...,...,...,...
408,2024-03-13 09:00:00,21.7135,1,1,1,enter,89.763981,0.000000,89.449807
409,2024-03-18 10:00:00,21.8522,1,0,0,out,90.337369,0.006388,90.021189
410,2024-03-18 17:00:00,23.0815,1,1,1,enter,90.337369,0.000000,90.021189
411,2024-03-25 11:00:00,24.1040,1,0,0,out,94.339274,0.044300,94.009086


Metricas

In [56]:
# creo dataframe para calculo de metricas
dfinputmetricas = dfindexdrawdawn[['datetime', 'state','index_sc', 'index','trade']]
# creo dataframe para almacenar metricas para cada conjunto de parametros


display(dfinputmetricas.head())

,datetime,state,index_sc,index,trade
0,2015-11-12 10:00:00,enter,100.000000,99.650000,0.000000
1,2015-11-13 10:00:00,out,98.764464,98.418788,-0.012355
2,2015-11-16 10:00:00,enter,98.764464,98.418788,0.000000
3,2015-11-19 15:00:00,out,102.840799,102.480856,0.041273
4,2015-11-23 08:00:00,enter,102.840799,102.480856,0.000000


Metrica Tir

Tir total anualizada

In [207]:
def tir_total_anualizada(dfinputmetricas):
    df = dfinputmetricas
    # Garante que datetime está no formato certo
    df['datetime'] = pd.to_datetime(df['datetime'])

    # Filtra enter e out
    df_enter = df[df['state'] == 'enter']
    df_out = df[df['state'] == 'out']

    # Verificação
    if df_enter.empty or df_out.empty:
        return None

    # Índice inicial e final
    idx_inicio = df_enter.iloc[0]['index']
    idx_fim = df_out.iloc[-1]['index']

    # Período completo entre primeira e última data do DataFrame
    dt_inicio_total = df['datetime'].min()
    dt_fim_total = df['datetime'].max()
    dias_total = (dt_fim_total - dt_inicio_total).days

    # Validação
    if dias_total <= 0 or idx_inicio == 0:
        return None

    # TIR anualizada com base no período total do df
    tirtotalanual = (idx_fim / idx_inicio) ** (365 / dias_total) - 1
    setirtotalanual = pd.Series({'tirtotalanual': tirtotalanual})

    return  setirtotalanual
    

In [211]:
setirtotalanual = tir_total_anualizada (dfinputmetricas)
display (setirtotalanual)

tirtotalanual   -0.006915
dtype: float64

In [64]:
#%%timeit
def tir_anuais_df (dfinputmetricas, dataini, datafim):

    # Exemplo do DataFrame original
    df = dfinputmetricas
    dataini = pd.to_datetime(dataini)
    datafim = pd.to_datetime(datafim)
    
    # Lista para novos registros
    novos_registros = []
    
    # Verifica se dataini deve ser adicionado
    if dataini < df.iloc[0]['datetime']:
        novos_registros.append({
            'datetime': dataini,
            'state': '',
            'index': df.iloc[0]['index']
        })
    
    # Verifica se datafim deve ser adicionado
    if datafim > df.iloc[-1]['datetime']:
        novos_registros.append({
            'datetime': datafim,
            'state': '',
            'index': df.iloc[-1]['index']
        })
    
    # Adiciona os registros e ordena
    df = pd.concat([pd.DataFrame(novos_registros), df], ignore_index=True)
    df = df.sort_values(by='datetime').reset_index(drop=True)
    
    # Determina os anos, excluindo o último ano
    ano_inicial = df['datetime'].min().year
    ano_final = (df['datetime'].max().year)
    
    # Gera os anos do intervalo EXCLUINDO o último ano
    anos_validos = range(ano_inicial, ano_final-1 )  # << ajuste aqui
    
    # Lista para os novos registros
    novos_registros = []
    
    for ano in anos_validos:
        fim_do_ano = pd.to_datetime(f'{ano}-12-31 23:59:59')
        df_antes = df[df['datetime'] < fim_do_ano]
        if not df_antes.empty:
            index_valor = df_antes.iloc[-1]['index']
            novos_registros.append({
                'datetime': fim_do_ano,
                'state': '',
                'index': index_valor
            })
    
    # Adiciona e organiza
    df = pd.concat([df, pd.DataFrame(novos_registros)], ignore_index=True)
    df = df.sort_values('datetime').reset_index(drop=True)
    
    # calcula o dataframe com as tir anuales ao fim do ano , com os anos incompletos anualizadas
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    #  consolidar por dia e manter o último registro
    df['date'] = df['datetime'].dt.date
    df_diario = df.sort_values('datetime').groupby('date', as_index=False).last()
    
    #  selecionar datas de fim de ano
    df_fim_ano = df_diario[
        (pd.to_datetime(df_diario['date']).dt.month == 12) &
        (pd.to_datetime(df_diario['date']).dt.day == 31)
    ].copy()
    
    #  calcular TIR entre pares de fim de ano
    resultados = []
    
    for i in range(1, len(df_fim_ano)):
        dt_inicio = pd.to_datetime(df_fim_ano.iloc[i - 1]['date'])
        dt_fim = pd.to_datetime(df_fim_ano.iloc[i]['date'])
        idx_inicio = df_fim_ano.iloc[i - 1]['index']
        idx_fim = df_fim_ano.iloc[i]['index']
        dias = (dt_fim - dt_inicio).days
    
        if dias > 0 and idx_inicio != 0:
            tir = (idx_fim / idx_inicio) ** (365 / dias) - 1
            resultados.append({
                'datetime': dt_fim,
                'tiranual': tir
            })
    
    #  adicionar último intervalo incompleto
    if not df_fim_ano.empty:
        dt_inicio = pd.to_datetime(df_fim_ano.iloc[-1]['date'])
        idx_inicio = df_fim_ano.iloc[-1]['index']
        dt_fim = pd.to_datetime(df_diario.iloc[-1]['date'])
        idx_fim = df_diario.iloc[-1]['index']
        dias = (dt_fim - dt_inicio).days
    
        if dias > 0 and idx_inicio != 0:
            tir = (idx_fim / idx_inicio) ** (365 / dias) - 1
            resultados.append({
                'datetime': dt_fim,
                'tiranual': tir
            })

    # criar DataFrame final
    dftiranual = pd.DataFrame(resultados)
    return dftiranual


In [66]:
dftiranual = tir_anuais_df (dfinputmetricas, dataini,datafim)
display (dftiranual)

,datetime,tiranual
0,2015-12-31,-0.147388
1,2016-12-31,0.073100
2,2017-12-31,0.221867
3,2018-12-31,0.058449
4,2019-12-31,-0.142933
5,2020-12-31,-0.285614
6,2021-12-31,0.160863
7,2022-12-31,0.048258
8,2024-04-03,0.055940


Tir Anuais Estatisticas

In [71]:
import pandas as pd

def tir_anuais_estat(dftiranual):
    tir = dftiranual['tiranual'].dropna()

    estatisticas = {
        'tiranualquant': tir.count(),
        'tiranualfirst': round(tir.iloc[0], 6),
        'tiranualmedia': round(tir.mean(), 6),
        'tiranualmax': round(tir.max(), 6),
        'tiranualmin': round(tir.min(), 6),
        'tiranualstd': round(tir.std(), 6)
    }

    return pd.Series(estatisticas)

In [73]:
setiranuaisestats = tir_anuais_estat(dftiranual)
print(setiranuaisestats)

tiranualquant    9.000000
tiranualfirst   -0.147388
tiranualmedia    0.004727
tiranualmax      0.221867
tiranualmin     -0.285614
tiranualstd      0.163113
dtype: float64


Metrica Trades

Trades Estatísticas

In [78]:
def trades_estatisticas(dfinputmetricas):
    df = dfinputmetricas.copy()
    trades = df["trade"].dropna()

    positivos = trades[trades > 0]
    negativos = trades[trades < 0]

    # Porcentagem de positivos
    porcentagem_pos = (len(positivos) / len(trades)) if len(trades) > 0 else 0

    ditradesestat = {
        "tradestot": len(trades),
        'tradefirst': round(trades.iloc[1], 6),
        "tradespositpor": round(porcentagem_pos, 6),
        "tradesposmedia": round(positivos.mean(), 6) if not positivos.empty else None,
        "tradesposstd": round(positivos.std(), 6) if not positivos.empty else None,
        "tradesposmax": round(positivos.max(), 6) if not positivos.empty else None,
        "tradesposmin": round(positivos.min(), 6) if not positivos.empty else None
    }

    return pd.Series(ditradesestat)

In [80]:
setradesestats = trades_estatisticas(dfinputmetricas)
print(setradesestats)

tradestot         413.000000
tradefirst         -0.012355
tradespositpor      0.159806
tradesposmedia      0.057078
tradesposstd        0.054829
tradesposmax        0.246621
tradesposmin        0.000795
dtype: float64


Metrica Drawdown

Drawdown Dataframe

In [84]:
def drawdowns_df (dfinputmetricas):
   
    df = dfinputmetricas
    df["datetime"] = pd.to_datetime(df["datetime"])
    serie = df["index"].dropna().reset_index(drop=True)
    datas = df["datetime"].reset_index(drop=True)

    drawdowns = []

    pico_idx = 0
    pico = serie[0]
    vale_idx = None
    valor_vale = None
    max_dd = 0

    for i in range(1, len(serie)):
        if serie[i] > pico:
            # Se recuperou acima do último pico: salvar ciclo anterior
            if vale_idx is not None and max_dd < 0:
                drawdowns.append({
                    "Data Pico": datas[pico_idx],
                    "Valor Pico": pico,
                    "Data Vale": datas[vale_idx],
                    "Valor Vale": valor_vale,
                    "Drawdown (%)": round(max_dd * 100, 2)
                })

            # Novo pico inicia novo ciclo
            pico = serie[i]
            pico_idx = i
            vale_idx = None
            max_dd = 0
        else:
            dd = (serie[i] - pico) / pico
            if dd < max_dd:
                max_dd = dd
                vale_idx = i
                valor_vale = serie[i]

    # Salva último ciclo, se aplicável
    if vale_idx is not None and max_dd < 0:
        drawdowns.append({
            "Data Pico": datas[pico_idx],
            "Valor Pico": pico,
            "Data Vale": datas[vale_idx],
            "Valor Vale": valor_vale,
            "Drawdown (%)": round(max_dd * 100, 2)
        })

    # Retorna os top N
    df_resultado = pd.DataFrame(drawdowns)
    return df_resultado.sort_values("Drawdown (%)").reset_index(drop=True)  

In [86]:
dfdrawdowns = drawdowns_df(dfinputmetricas)
display (dfdrawdowns)

,Data Pico,Valor Pico,Data Vale,Valor Vale,Drawdown (%)
0,2019-07-31 16:00:00,124.937454,2021-03-05 10:00:00,66.749733,-46.57
1,2015-11-19 15:00:00,102.480856,2016-04-11 15:00:00,77.925056,-23.96
2,2019-02-06 13:00:00,121.304726,2019-04-11 11:00:00,97.436077,-19.68
3,2017-06-02 14:00:00,112.498600,2017-11-08 14:00:00,95.353084,-15.24
4,2018-01-18 09:00:00,118.372928,2018-02-08 14:00:00,111.276594,-5.99
5,2017-04-07 09:00:00,106.470462,2017-04-26 16:00:00,103.066664,-3.20
6,2017-05-10 14:00:00,108.212797,2017-05-18 09:00:00,104.791444,-3.16
7,2017-12-12 10:00:00,113.768726,2017-12-22 09:00:00,111.423541,-2.06
8,2015-11-12 10:00:00,99.650000,2015-11-13 10:00:00,98.418788,-1.24


Drawdown Estatisticas

In [89]:

def drawdowns_estat(dfdrawdowns):
    dd = dfdrawdowns['Drawdown (%)'].dropna()  # Filtra nulos, se houver

    estatisticas = {
        'drawdfirst': round(dd.iloc[0], 6),
        'drawdtot': dd.count(),
        'drawdmedia': round(dd.mean(), 2),
        'drawdmaximo': round(dd.max(), 2),
        'drawdminimo': round(dd.min(), 2),
        'drawdstd': round(dd.std(), 2)
    }

    return pd.Series(estatisticas)

In [91]:
sedrawdownsestats = drawdowns_estat(dfdrawdowns)
display(sedrawdownsestats)

drawdfirst    -46.57
drawdtot        9.00
drawdmedia    -13.46
drawdmaximo    -1.24
drawdminimo   -46.57
drawdstd       14.98
dtype: float64

Metrica Dias Out

Dias Out Dataframe

In [97]:
def dias_out_df (df, coluna="index_sc"):
    
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df[df[coluna].notna()].reset_index(drop=True)

    variacao = df[coluna].diff()
    grupos = (variacao != 0).cumsum()

    agrupado = df.groupby(grupos)
    periodos_estaticos = []

    for _, grupo in agrupado:
        if len(grupo) > 1 and grupo[coluna].nunique() == 1:
            duracao_dias = (grupo["datetime"].iloc[-1] - grupo["datetime"].iloc[0]).days
            periodos_estaticos.append({                
                "Data Início": grupo["datetime"].iloc[0],
                "Data Fim": grupo["datetime"].iloc[-1],
                "difdias": duracao_dias,
                "Valor index": grupo[coluna].iloc[0]
            })

    dfperiodos = pd.DataFrame(periodos_estaticos)
    dfperiodos = dfperiodos.query("difdias != 0").copy()
    dfperiodos = dfperiodos.reset_index(drop=True)
    
    return dfperiodos

In [99]:
dfdiasout = dias_out_df (dfinputmetricas)
display(dfdiasout.head())


,Data Início,Data Fim,difdias,Valor index
0,2015-11-13 10:00:00,2015-11-16 10:00:00,3,98.764464
1,2015-11-19 15:00:00,2015-11-23 08:00:00,3,102.840799
2,2015-11-23 09:00:00,2015-11-30 15:00:00,7,94.116234
3,2015-12-03 09:00:00,2015-12-08 09:00:00,5,91.610145
4,2015-12-14 11:00:00,2015-12-16 16:00:00,2,89.806591


Dias Out Estatisticas

In [102]:
def dias_out_estats(dfdiasout):
    dias = dfdiasout['difdias'].dropna()  # Remove valores nulos, se houver

    estatisticas = {
        'diasoutfirst': round(dias.iloc[0], 6),
        'diasouttot': dias.sum(),
        'diasoutmedia': round(dias.mean(), 2),
        'diasoutmax': dias.max(),
        'diasoutmin': dias.min(),
        'diasoutstd': round(dias.std(), 2)
    }

    return pd.Series(estatisticas)

In [106]:
sediasoutestats = dias_out_estats(dfdiasout)
display(sediasoutestats)

diasoutfirst       3.00
diasouttot      2368.00
diasoutmedia      17.41
diasoutmax       331.00
diasoutmin         1.00
diasoutstd        44.82
dtype: float64

Metrica StopSys Dataframe

In [111]:
def stopsys_df (dfinputmetricas) :   
    # Garante que a coluna 'datetime' esteja no formato correto
    dfinputmetricas['datetime'] = pd.to_datetime(dfinputmetricas['datetime'])
    
    # Filtra os registros onde state == 'stopsys'
    dfstopsys = dfinputmetricas[dfinputmetricas['state'] == 'stopsys'][['datetime', 'state', 'index']].copy()
    
    #  Ordena por datetime
    dfstopsys = dfstopsys.sort_values('datetime').reset_index(drop=True)
    return dfstopsys


In [113]:
dfstopsys = stopsys_df (dfinputmetricas)
display (dfstopsys)

,datetime,state,index
0,2015-12-02 09:00:00,stopsys,91.438290
1,2016-01-15 09:00:00,stopsys,80.601087
2,2017-11-02 13:00:00,stopsys,98.417176
3,2019-03-12 13:00:00,stopsys,108.629329
4,2019-04-11 11:00:00,stopsys,97.436077
5,2019-08-12 05:00:00,stopsys,112.132882
6,2020-03-06 13:00:00,stopsys,98.840254
7,2020-06-22 11:00:00,stopsys,87.673940
8,2020-09-25 08:00:00,stopsys,83.551626
9,2020-10-09 13:00:00,stopsys,73.499568


In [115]:

def stopsys_estat(dfstopsys):
    df = dfstopsys.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime').reset_index(drop=True)

    index_values = df['index'].dropna()

    estatisticas = {
        'stopsysfirst': round(index_values.iloc[0], 6),
        'stopsysquant': index_values.count(),
        'stopsysmedia': round(index_values.mean(), 6),
        'stopsysmaximo': round(index_values.max(), 6),
        'stopsysminimo': round(index_values.min(), 6),
        'stopsystd': round(index_values.std(), 6)
    }

    return pd.Series(estatisticas)

In [117]:
sestopsysestats = stopsys_estat(dfstopsys)
display (sestopsysestats )

stopsysfirst      91.438290
stopsysquant      19.000000
stopsysmedia      89.501658
stopsysmaximo    112.132882
stopsysminimo     69.154694
stopsystd         12.650638
dtype: float64

In [217]:
#del dfmetricas
#dfmetricas = None

In [219]:
import pandas as pd

def atualizar_metricas(series_list, dfmetricas):
    """
    Atualiza o DataFrame dfmetricas com uma nova linha a partir de várias Series.
    Se dfmetricas não existir, cria com a primeira linha.
    Cada chamada adiciona uma nova linha, mesmo com mesmos nomes de índice.

    Parâmetros:
        series_list: lista de pandas.Series
        dfmetricas: pandas.DataFrame ou None

    Retorno:
        dfmetricas atualizado
    """
    # Junta todas as Series em uma lista de valores
    valores = pd.concat(series_list).values

    # Cria DataFrame com uma nova linha usando os nomes das colunas apenas da primeira chamada
    if dfmetricas is None or dfmetricas.empty:
        colunas = pd.concat(series_list).index
        dfmetricas = pd.DataFrame([valores], columns=colunas)
    else:
        nova_linha_df = pd.DataFrame([valores], columns=dfmetricas.columns)
        dfmetricas = pd.concat([dfmetricas, nova_linha_df], ignore_index=True)

    return dfmetricas

In [221]:
dfmetricas = atualizar_metricas([setirtotalanual,setiranuaisestats,
    setradesestats,sedrawdownsestats,sediasoutestats],dfmetricas)
display (dfmetricas)


,tirtotalanual,tiranualquant,tiranualfirst,tiranualmedia,tiranualmax,tiranualmin,tiranualstd,tradestot,tradefirst,tradespositpor,...,drawdmedia,drawdmaximo,drawdminimo,drawdstd,diasoutfirst,diasouttot,diasoutmedia,diasoutmax,diasoutmin,diasoutstd
0,-0.006915,9.0,-0.147388,0.004727,0.221867,-0.285614,0.163113,413.0,-0.012355,0.159806,...,-13.46,-1.24,-46.57,14.98,3.0,2368.0,17.41,331.0,1.0,44.82
